In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


print("All imports successful")

Matplotlib is building the font cache; this may take a moment.


All imports successful


In [3]:
mh = pd.read_csv('13100465.csv')       # mental health table
unemp = pd.read_csv('14100017.csv')    # unemployment table

# first look at structure
print(mh.shape)
print(mh.columns.tolist())
mh.head()



(160992, 18)
['REF_DATE', 'GEO', 'DGUID', 'Age group', 'Gender', 'Indicators', 'Characteristics', 'UOM', 'UOM_ID', 'SCALAR_FACTOR', 'SCALAR_ID', 'VECTOR', 'COORDINATE', 'VALUE', 'STATUS', 'SYMBOL', 'TERMINATED', 'DECIMALS']


,REF_DATE,GEO,DGUID,Age group,Gender,Indicators,Characteristics,UOM,UOM_ID,SCALAR_FACTOR,SCALAR_ID,VECTOR,COORDINATE,VALUE,STATUS,SYMBOL,TERMINATED,DECIMALS
0,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Major depressive episode, life",Number of persons,Number,223,units,0,v73953257,1.1.1.5.1,3037049.0,NaN,NaN,NaN,0
1,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Major depressive episode, life","Low 95% confidence interval, number of persons",Number,223,units,0,v73953258,1.1.1.5.2,2913405.0,NaN,NaN,NaN,0
2,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Major depressive episode, life","High 95% confidence interval, number of persons",Number,223,units,0,v73953259,1.1.1.5.3,3160692.0,NaN,NaN,NaN,0
3,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Major depressive episode, life",Coefficient of variation for number of persons,Number,223,units,0,v73953260,1.1.1.5.4,21.0,NaN,NaN,NaN,0
4,2002,Canada,2021A000011124,"Total, 15 years and over","Total, gender of person","Major depressive episode, life",Percent,Percent,239,units,0,v73953261,1.1.1.5.5,12.2,NaN,NaN,NaN,1


In [17]:
mh['REF_DATE'].unique()

array([2002, 2012, 2022])

In [16]:
print(unemp.shape)
print(unemp.columns.tolist())
unemp.head()

unemp["REF_DATE"].unique()

(4414080, 17)
['REF_DATE', 'GEO', 'DGUID', 'Labour force characteristics', 'Gender', 'Age group', 'UOM', 'UOM_ID', 'SCALAR_FACTOR', 'SCALAR_ID', 'VECTOR', 'COORDINATE', 'VALUE', 'STATUS', 'SYMBOL', 'TERMINATED', 'DECIMALS']


<StringArray>
['1976-01', '1976-02', '1976-03', '1976-04', '1976-05', '1976-06', '1976-07',
 '1976-08', '1976-09', '1976-10',
 ...
 '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05',
 '2026-06', '2026-07', '2026-08']
Length: 608, dtype: str

# Unemployment and mental health in Canada, 2002–2022

**Question:** Is the unemployment rate associated with mental-health outcomes, and does this association vary by gender and province?

**Plan**
1. Filter both tables and build one row per province × gender × age group × survey year
2. Check the raw correlation and the age confounder
3. Fit three regression models (raw → controls → gender/province interactions)
4. Robustness checks
5. Province-level analysis (province totals, fixed effects, 2002 vs 2022)
6. Export Tableau-ready CSVs

In [ ]:
import statsmodels.formula.api as smf
import os

PROVINCES = ['Newfoundland and Labrador', 'Prince Edward Island', 'Nova Scotia', 'New Brunswick',
             'Quebec', 'Ontario', 'Manitoba', 'Saskatchewan', 'Alberta', 'British Columbia']
AGES = ['15 to 24 years', '25 to 44 years', '45 to 64 years', '65 years and over']
GENDERS = ['Men+', 'Women+']
YEARS = [2002, 2012, 2022]
OUTCOME = 'Major depressive episode, 12 months'   # main outcome

## 1. Data preparation
**Mental health:** keep the *Percent* rows for provinces, the four age groups and men/women, plus the STATUS quality flag (E = use with caution, F = too unreliable, x = suppressed).

In [ ]:
def get_mh(indicator):
    s = mh[(mh['Indicators'] == indicator) &
           (mh['Characteristics'] == 'Percent') &
           (mh['GEO'].isin(PROVINCES)) &
           (mh['Age group'].isin(AGES)) &
           (mh['Gender'].isin(GENDERS))]
    return (s.rename(columns={'REF_DATE': 'year', 'GEO': 'geo', 'Age group': 'age',
                              'Gender': 'gender', 'VALUE': 'mh', 'STATUS': 'status'})
             [['year', 'geo', 'age', 'gender', 'mh', 'status']])

mh_dep = get_mh(OUTCOME)
print('Cells in design:', len(mh_dep))
print('Published values:', mh_dep['mh'].notna().sum())
print(mh_dep['status'].value_counts(dropna=False))

**Unemployment:** keep the unemployment rate for 2002, 2012 and 2022 and average the 12 months into an annual rate.

In [ ]:
u = unemp[unemp['Labour force characteristics'] == 'Unemployment rate'].copy()
u['year'] = u['REF_DATE'].str[:4].astype(int)
u = u[u['year'].isin(YEARS)]

u_groups = (u[u['GEO'].isin(PROVINCES) & u['Age group'].isin(AGES) & u['Gender'].isin(GENDERS)]
            .groupby(['year', 'GEO', 'Age group', 'Gender'], as_index=False)['VALUE'].mean()
            .rename(columns={'GEO': 'geo', 'Age group': 'age', 'Gender': 'gender', 'VALUE': 'unemp'}))
print(u_groups.shape)
u_groups.head()

In [ ]:
def build(indicator):
    d = get_mh(indicator).merge(u_groups, on=['year', 'geo', 'age', 'gender'])
    return d.dropna(subset=['mh', 'unemp'])

df = build(OUTCOME)
print('Analysis rows:', len(df))
print(df.groupby('year').size())
print(df.groupby('age').size())
df.head()

## 2. Raw correlation and the age confounder
Age drives both variables: young people have the highest unemployment **and** the highest depression. Within each age group the correlation turns negative.

In [ ]:
print('Overall r =', round(df['mh'].corr(df['unemp']), 3))
print('\nr within each age group:')
print(df.groupby('age')[['mh', 'unemp']].corr().unstack().iloc[:, 1].round(3))
print('\nMeans by age group:')
print(df.groupby('age')[['mh', 'unemp']].mean().round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=df, x='unemp', y='mh', hue='age', ax=ax)
sns.regplot(data=df, x='unemp', y='mh', scatter=False, color='black', ax=ax,
            line_kws={'linestyle': '--'}, label='Overall trend')
for age, g in df.groupby('age'):
    if len(g) > 5:
        sns.regplot(data=g, x='unemp', y='mh', scatter=False, ci=None, ax=ax)
ax.set_xlabel('Unemployment rate (%)')
ax.set_ylabel('Major depressive episode, past 12 months (%)')
ax.set_title('Positive overall, negative within each age group')
plt.tight_layout(); plt.show()

## 3. Three regression models
- **Model 1:** depression ~ unemployment
- **Model 2:** + gender, age, province, year (controls)
- **Model 3:** + unemployment × gender and unemployment × province (does the slope vary?)

Reference groups: men, Ontario, 15–24, 2002. HC3 robust standard errors.

In [ ]:
PROV = "C(geo, Treatment('Ontario'))"
GEN = "C(gender, Treatment('Men+'))"

def fit_models(d):
    m1 = smf.ols('mh ~ unemp', data=d).fit(cov_type='HC3')
    m2 = smf.ols(f'mh ~ unemp + {GEN} + C(age) + {PROV} + C(year)', data=d).fit(cov_type='HC3')
    m3 = smf.ols(f'mh ~ unemp * {GEN} + unemp * {PROV} + C(age) + C(year)', data=d).fit(cov_type='HC3')
    return m1, m2, m3

m1, m2, m3 = fit_models(df)

summary = pd.DataFrame({
    'beta_unemp': [m.params['unemp'] for m in (m1, m2, m3)],
    'se': [m.bse['unemp'] for m in (m1, m2, m3)],
    'p': [m.pvalues['unemp'] for m in (m1, m2, m3)],
    'R2': [m.rsquared for m in (m1, m2, m3)],
    'adj_R2': [m.rsquared_adj for m in (m1, m2, m3)],
    'N': [int(m.nobs) for m in (m1, m2, m3)],
}, index=['Model 1: raw', 'Model 2: + controls', 'Model 3: + interactions']).round(3)
summary

In [ ]:
print(m2.summary())

In [ ]:
# Joint tests: does the unemployment slope differ by gender? by province?
print(m3.wald_test_terms(scalar=True))

### Slopes by gender and by province (from Model 3)

In [ ]:
def combined_slope(model, extra_term=None):
    w = pd.Series(0.0, index=model.params.index)
    w['unemp'] = 1
    if extra_term is not None and extra_term in w.index:
        w[extra_term] = 1
    t = model.t_test(w.values)
    lo, hi = t.conf_int()[0]
    return float(np.squeeze(t.effect)), lo, hi, float(np.squeeze(t.pvalue))

gender_term = f'unemp:{GEN}[T.Women+]'
gender_slopes = pd.DataFrame([('Men', *combined_slope(m3)),
                              ('Women', *combined_slope(m3, gender_term))],
                             columns=['group', 'slope', 'ci_low', 'ci_high', 'p']).round(3)
print(gender_slopes)

prov_rows = []
for p in PROVINCES:
    term = f'unemp:{PROV}[T.{p}]'
    if p != 'Ontario' and term not in m3.params.index:
        continue   # province with no data for this outcome (e.g. PEI)
    s, lo, hi, _ = combined_slope(m3, term)
    prov_rows.append((p, s, lo, hi, m3.pvalues.get(term, np.nan)))
prov_slopes = (pd.DataFrame(prov_rows, columns=['province', 'slope', 'ci_low', 'ci_high', 'p_vs_ontario'])
               .sort_values('slope').round(3))
prov_slopes

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
y = range(len(prov_slopes))
ax.errorbar(prov_slopes['slope'], y,
            xerr=[prov_slopes['slope'] - prov_slopes['ci_low'], prov_slopes['ci_high'] - prov_slopes['slope']],
            fmt='o', color='grey', ecolor='lightgrey', capsize=3)
ax.axvline(0, color='black', linewidth=1)
ax.set_yticks(list(y)); ax.set_yticklabels(prov_slopes['province'])
ax.set_xlabel('Slope: % depression per 1-point rise in unemployment (men, 95% CI)')
ax.set_title('No province differs from Ontario (joint test p = %.2f)'
             % m3.wald_test_terms(scalar=True).summary_frame().loc[f'unemp:{PROV}', 'P>chi2'])
plt.tight_layout(); plt.show()

## 4. Robustness checks

In [ ]:
def m2_beta(d, label):
    m = smf.ols(f'mh ~ unemp + {GEN} + C(age) + {PROV} + C(year)', data=d).fit(cov_type='HC3')
    return label, round(m.params['unemp'], 3), round(m.pvalues['unemp'], 3), int(m.nobs)

df_poor = build('Perceived mental health, fair or poor')
_, _, m3_poor = fit_models(df_poor)

robust = pd.DataFrame([
    m2_beta(df, 'Main: depression'),
    m2_beta(df[df['age'] != '65 years and over'], 'Drop 65+'),
    m2_beta(df_poor, 'Outcome: fair or poor mental health'),
], columns=['check', 'beta_unemp', 'p', 'N'])
print(robust)
print('\nFair/poor outcome, Model 3 joint tests:')
print(m3_poor.wald_test_terms(scalar=True).summary_frame().filter(like='unemp', axis=0))

## 5. Across provinces (province totals)
Province totals (all ages, both genders) are far more reliable than the small cells. Two comparisons:
- **Across provinces in the same year** (year fixed effects)
- **Within a province over time** (province + year fixed effects)

In [ ]:
INDICATORS = {
    'dep': 'Major depressive episode, 12 months',
    'poor': 'Perceived mental health, fair or poor',
    'good': 'Perceived mental health, very good or excellent',
    'suic': 'Suicidal thoughts, 12 months',
}
tot = mh[(mh['Characteristics'] == 'Percent') & mh['GEO'].isin(PROVINCES) &
         (mh['Age group'] == 'Total, 15 years and over') &
         (mh['Gender'] == 'Total, gender of person') &
         mh['Indicators'].isin(INDICATORS.values())]
prov = (tot.pivot_table(index=['GEO', 'REF_DATE'], columns='Indicators', values='VALUE')
           .rename(columns={v: k for k, v in INDICATORS.items()}).reset_index())

u_tot = (u[u['GEO'].isin(PROVINCES) & (u['Age group'] == '15 years and over') & (u['Gender'] == 'Total - Gender')]
         .groupby(['GEO', 'year'], as_index=False)['VALUE'].mean()
         .rename(columns={'year': 'REF_DATE', 'VALUE': 'unemp'}))
prov = prov.merge(u_tot, on=['GEO', 'REF_DATE'])
prov.round(1)

In [ ]:
rows = []
for k in INDICATORS:
    d = prov.dropna(subset=[k])
    a = smf.ols(f'{k} ~ unemp + C(REF_DATE)', data=d).fit(cov_type='HC3')
    b = smf.ols(f'{k} ~ unemp + C(REF_DATE) + C(GEO)', data=d).fit(cov_type='HC3')
    rows.append((INDICATORS[k], len(d),
                 a.params['unemp'], a.pvalues['unemp'],
                 b.params['unemp'], b.pvalues['unemp']))
fe = pd.DataFrame(rows, columns=['outcome', 'N',
                                 'beta_across_provinces', 'p_across',
                                 'beta_within_province', 'p_within']).round(3)
fe

In [ ]:
# 2002 vs 2022: fair or poor mental health by province
wide = prov.pivot(index='GEO', columns='REF_DATE', values=['poor', 'unemp'])
change = pd.DataFrame({
    'poor_2002': wide[('poor', 2002)], 'poor_2022': wide[('poor', 2022)],
    'unemp_2002': wide[('unemp', 2002)], 'unemp_2022': wide[('unemp', 2022)],
}).dropna()
change['poor_change'] = change['poor_2022'] - change['poor_2002']
change['unemp_change'] = change['unemp_2022'] - change['unemp_2002']
change = change.sort_values('poor_2022')
print('r(unemployment change, mental health change) =',
      round(change['unemp_change'].corr(change['poor_change']), 2))

fig, ax = plt.subplots(figsize=(8, 5))
y = range(len(change))
ax.hlines(y, change['poor_2002'], change['poor_2022'], color='lightgrey', linewidth=2)
ax.scatter(change['poor_2002'], y, color='grey', label='2002', zorder=3)
ax.scatter(change['poor_2022'], y, color='tab:blue', label='2022', zorder=3)
ax.set_yticks(list(y)); ax.set_yticklabels(change.index)
ax.set_xlabel('Fair or poor mental health (%)')
ax.set_title('Mental health worsened in every province while unemployment mostly fell')
ax.legend(); plt.tight_layout(); plt.show()
change.round(1)

## 6. Export Tableau-ready CSVs

In [ ]:
os.makedirs('results/tableau', exist_ok=True)
df.to_csv('results/tableau/3_group_cells_depression.csv', index=False)
prov.to_csv('results/tableau/1_province_year_wide.csv', index=False)
prov_slopes.to_csv('results/tableau/4_province_slopes_model3.csv', index=False)
summary.reset_index(names='model').to_csv('results/tableau/5_model_summary.csv', index=False)
print(os.listdir('results/tableau'))

## 7. Who was hit hardest in 2022
National estimates by age and gender (much more reliable than province cells). Young women deteriorated most on every measure.

In [ ]:
YOUTH_INDICATORS = ['Perceived mental health, fair or poor', 'Major depressive episode, 12 months',
                    'Generalized anxiety disorder, 12 months', 'Suicidal thoughts, 12 months']
nat = mh[(mh['GEO'] == 'Canada') & (mh['Characteristics'] == 'Percent') &
         mh['Age group'].isin(AGES) & mh['Gender'].isin(GENDERS) &
         mh['Indicators'].isin(YOUTH_INDICATORS)]
nat = (nat.rename(columns={'REF_DATE': 'Year', 'Age group': 'Age group', 'VALUE': 'Percent'})
          [['Year', 'Indicators', 'Age group', 'Gender', 'Percent']].dropna())
nat['Gender'] = nat['Gender'].str.replace('+', '', regex=False)

# 2012 -> 2022 change for each indicator, age and gender
chg = nat.pivot_table(index=['Indicators', 'Age group', 'Gender'], columns='Year', values='Percent')
chg['change_2012_2022'] = chg[2022] - chg[2012]
chg.round(1)

In [ ]:
fp = nat[nat['Indicators'] == 'Perceived mental health, fair or poor']
fig, ax = plt.subplots(figsize=(9, 5.5))
for (age, gender), g in fp.groupby(['Age group', 'Gender']):
    g = g.sort_values('Year')
    if age == '15 to 24 years' and gender == 'Women':
        ax.plot(g['Year'], g['Percent'], color='tab:blue', linewidth=3, marker='o', zorder=3)
        ax.annotate(f'Women 15-24: {g.Percent.iloc[-1]:.1f}%', (2022, g.Percent.iloc[-1]),
                    xytext=(8, 0), textcoords='offset points', color='tab:blue', fontweight='bold', va='center')
    elif age == '15 to 24 years':
        ax.plot(g['Year'], g['Percent'], color='dimgrey', linewidth=2, marker='o', zorder=2)
        ax.annotate(f'Men 15-24: {g.Percent.iloc[-1]:.1f}%', (2022, g.Percent.iloc[-1]),
                    xytext=(8, 0), textcoords='offset points', color='dimgrey', va='center')
    else:
        ax.plot(g['Year'], g['Percent'], color='lightgrey', linewidth=1.5, marker='o', zorder=1)
ax.set_xticks([2002, 2012, 2022]); ax.set_xlim(2000, 2028); ax.set_ylim(0, 32)
ax.set_ylabel('Fair or poor mental health (%)')
ax.set_title('Nearly 1 in 3 young women rated their mental health fair or poor in 2022\n(grey = other age x gender groups)')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
# Tableau export: one row per indicator x age x gender x year
nat.to_csv('results/tableau/6_age_gender_trends_national.csv', index=False)
print(len(nat), 'rows saved')

## Summary of findings
1. **Raw link is weak** (r ≈ 0.16, Model 1 p ≈ 0.10) and **confounded by age**: within each age group the correlation is negative.
2. **With controls, no association** (Model 2: β ≈ −0.15, p ≈ 0.46). Gender, age and year explain most of the variation (R² 0.03 → 0.71).
3. **No variation by gender** (interaction p ≈ 0.67) **or province** (joint p ≈ 0.20).
4. **Across provinces the sign reverses:** higher-unemployment provinces (e.g. Newfoundland and Labrador) report *better* mental health, but this disappears with province fixed effects, so it reflects stable provincial differences, not unemployment.
5. **2022:** fair or poor mental health doubled nationally (6.9% → 15.3%) while unemployment fell (7.7% → 5.3%), most likely a pandemic effect.
6. **Caveat:** only 87 of 240 depression cells are published, and most are flagged E, so "no detectable association" is not proof of "no effect".
7. **Who was hit hardest:** women aged 15–24 rated their mental health fair or poor at 29.0% in 2022, almost triple 2012 (10.1%); depression 9.0% → 18.4%, anxiety 3.8% → 11.9%. People 65+ barely changed.